# 04. OD 매트릭스 분석

DC_TBYXD012의 승차/하차 행정동코드(RIDE_A_CD, ALIGHT_A_CD)로 OD 매트릭스를 만들고
상위 OD, 시간대/요일 패턴, 유입·유출 불균형, 거리 분포를 분석한다.

> 전부 청크 누적 집계로 처리 — 6억건 규모를 메모리에 올리지 않는다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform
import warnings
warnings.filterwarnings('ignore')

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)
import seaborn as sns

In [ ]:
import gc, psutil, os

def mem_usage(tag=''):
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'[MEM {tag}] {gb:.2f} GB')

CHUNK_SIZE = 1_000_000
mem_usage('start')

## 1. 청크 집계: OD쌍 / 시간대·요일 / 유입·유출 / 거리

In [ ]:
D012_PATH = './DC_TBYXD012.csv'
usecols = ['RIDE_DTIME', 'PAY_AMT', 'RIDE_DIST', 'RIDE_A_CD', 'ALIGHT_A_CD']
dtypes  = {'RIDE_DTIME': str, 'PAY_AMT': 'float64', 'RIDE_DIST': 'float64',
           'RIDE_A_CD': str, 'ALIGHT_A_CD': str}

def timeband(h):
    if 7 <= h <= 9:   return '출근(7-9)'
    if 11 <= h <= 13: return '점심(11-13)'
    if 17 <= h <= 19: return '퇴근(17-19)'
    if h >= 23 or h <= 4: return '심야(23-4)'
    return '기타'

od_parts, tb_parts, dow_parts = [], [], []
outflow = {}; inflow = {}
total = 0
for chunk in pd.read_csv(D012_PATH, usecols=usecols, dtype=dtypes, chunksize=CHUNK_SIZE):
    rd = pd.to_datetime(chunk['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    m = rd.notna(); chunk = chunk[m].copy(); rd = rd[m]
    chunk['hour'] = rd.dt.hour
    chunk['day_name'] = rd.dt.day_name()
    chunk['timeband'] = chunk['hour'].map(timeband)
    chunk['OD'] = chunk['RIDE_A_CD'] + '->' + chunk['ALIGHT_A_CD']
    total += len(chunk)

    od_parts.append(chunk.groupby(['RIDE_A_CD','ALIGHT_A_CD']).agg(
        trip_count=('PAY_AMT','size'), dist_sum=('RIDE_DIST','sum'),
        fare_sum=('PAY_AMT','sum')).reset_index())
    tb_parts.append(chunk.groupby(['OD','timeband']).size().reset_index(name='count'))
    dow_parts.append(chunk.groupby(['OD','day_name']).size().reset_index(name='count'))
    for k, v in chunk.groupby('RIDE_A_CD').size().items():   outflow[k] = outflow.get(k, 0) + v
    for k, v in chunk.groupby('ALIGHT_A_CD').size().items(): inflow[k]  = inflow.get(k, 0) + v
    del chunk, rd; gc.collect()

od_matrix = pd.concat(od_parts).groupby(['RIDE_A_CD','ALIGHT_A_CD']).agg(
    trip_count=('trip_count','sum'), dist_sum=('dist_sum','sum'), fare_sum=('fare_sum','sum')).reset_index()
od_matrix['avg_dist'] = od_matrix['dist_sum'] / od_matrix['trip_count']
od_matrix['avg_fare'] = od_matrix['fare_sum'] / od_matrix['trip_count']
od_matrix['OD'] = od_matrix['RIDE_A_CD'] + '->' + od_matrix['ALIGHT_A_CD']
tb_od  = pd.concat(tb_parts).groupby(['OD','timeband'])['count'].sum().reset_index()
dow_od = pd.concat(dow_parts).groupby(['OD','day_name'])['count'].sum().reset_index()
del od_parts, tb_parts, dow_parts; gc.collect()

print(f"전체 {total:,}건, OD쌍 {len(od_matrix):,}개")
mem_usage('after load')

## 2. OD 매트릭스 히트맵 (상위 행정동)

In [ ]:
top_o = od_matrix.groupby('RIDE_A_CD')['trip_count'].sum().nlargest(15).index
top_d = od_matrix.groupby('ALIGHT_A_CD')['trip_count'].sum().nlargest(15).index
codes = list(dict.fromkeys(list(top_o) + list(top_d)))[:15]
piv = od_matrix[od_matrix['RIDE_A_CD'].isin(codes) & od_matrix['ALIGHT_A_CD'].isin(codes)] \
    .pivot_table(index='RIDE_A_CD', columns='ALIGHT_A_CD', values='trip_count', fill_value=0).astype(int)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(piv, annot=True, fmt='d', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('OD 매트릭스 (상위 행정동)', fontweight='bold')
ax.set_xlabel('하차 행정동'); ax.set_ylabel('승차 행정동')
plt.tight_layout(); plt.show()
print(f"총 통행량: {od_matrix['trip_count'].sum():,}")

## 3. 상위 OD 쌍 Top 20

In [ ]:
top20 = od_matrix.nlargest(20, 'trip_count')
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(range(len(top20)), top20['trip_count'], color='steelblue', edgecolor='black', lw=0.5)
ax.set_yticks(range(len(top20))); ax.set_yticklabels(top20['OD'], fontsize=9)
ax.invert_yaxis(); ax.set_xlabel('통행 건수'); ax.set_title('상위 OD 쌍 Top 20', fontweight='bold')
for i, v in enumerate(top20['trip_count']): ax.text(v, i, f' {v:,.0f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

## 4. 시간대별 OD 패턴

In [ ]:
top_od = top20['OD'].tolist()[:10]
order = ['출근(7-9)','점심(11-13)','퇴근(17-19)','심야(23-4)','기타']
tp = tb_od[tb_od['OD'].isin(top_od)].pivot_table(index='OD', columns='timeband', values='count', fill_value=0)
tp = tp.reindex(columns=[c for c in order if c in tp.columns]).astype(int)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(tp, annot=True, fmt='d', cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('시간대별 주요 OD 통행량', fontweight='bold'); ax.set_xlabel('시간대'); ax.set_ylabel('OD')
plt.tight_layout(); plt.show()

## 5. 요일별 주요 OD

In [ ]:
dorder = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dkr = dict(zip(dorder, ['월','화','수','목','금','토','일']))
dp = dow_od[dow_od['OD'].isin(top_od)].pivot_table(index='OD', columns='day_name', values='count', fill_value=0)
dp = dp.reindex(columns=[d for d in dorder if d in dp.columns]).astype(int)
dp.columns = [dkr[c] for c in dp.columns]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(dp, annot=True, fmt='d', cmap='Greens', ax=ax, linewidths=0.5)
ax.set_title('요일별 주요 OD 통행량', fontweight='bold'); ax.set_xlabel('요일'); ax.set_ylabel('OD')
plt.tight_layout(); plt.show()

## 6. 유입/유출 불균형

In [ ]:
bal = pd.DataFrame({'outflow': pd.Series(outflow), 'inflow': pd.Series(inflow)}).fillna(0).astype(int)
bal['net'] = bal['inflow'] - bal['outflow']
bal = bal.sort_values('net')
plot = pd.concat([bal.nsmallest(10, 'net'), bal.nlargest(10, 'net')])
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(range(len(plot)), plot['net'], color=['#e74c3c' if v<0 else '#2ecc71' for v in plot['net']],
        edgecolor='black', lw=0.5)
ax.set_yticks(range(len(plot))); ax.set_yticklabels(plot.index, fontsize=9)
ax.axvline(0, color='black', lw=0.8); ax.set_xlabel('순 유입량 (유입-유출)')
ax.set_title('행정동별 유입/유출 불균형', fontweight='bold')
plt.tight_layout(); plt.show()

## 7. OD 거리 분포

In [ ]:
min_trips = max(5, od_matrix['trip_count'].quantile(0.5))
valid = od_matrix[od_matrix['trip_count'] >= min_trips].sort_values('avg_dist', ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[1].hist(valid['avg_dist'], bins=30, color='coral', edgecolor='black', lw=0.5)
axes[1].set_title(f'OD쌍별 평균거리 분포 (N>={int(min_trips)})'); axes[1].set_xlabel('평균거리(m)'); axes[1].set_ylabel('OD쌍 수')
axes[0].hist(od_matrix['avg_dist'].dropna(), bins=50, color='steelblue', edgecolor='black', lw=0.5)
axes[0].set_title('전체 OD 평균거리 분포'); axes[0].set_xlabel('평균거리(m)'); axes[0].set_ylabel('OD쌍 수')
plt.tight_layout(); plt.show()
print('장거리 Top5:'); print(valid.head()[['OD','avg_dist','trip_count']].to_string(index=False))

## 8. 요약

In [ ]:
summary = {
    '전체 통행 건수': f"{od_matrix['trip_count'].sum():,.0f}",
    '고유 OD 쌍 수': f"{len(od_matrix):,}",
    '최다 OD 쌍': top20.iloc[0]['OD'],
    '최다 OD 건수': f"{top20.iloc[0]['trip_count']:,.0f}",
    '평균 운행거리(m)': f"{od_matrix['avg_dist'].mean():,.0f}",
    '유입 최다': bal['net'].idxmax(),
    '유출 최다': bal['net'].idxmin(),
}
print('=== OD 매트릭스 요약 ===')
for k, v in summary.items(): print(f'  {k}: {v}')